In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/deep-past-initiative-machine-translation/sample_submission.csv
/kaggle/input/competitions/deep-past-initiative-machine-translation/bibliography.csv
/kaggle/input/competitions/deep-past-initiative-machine-translation/publications.csv
/kaggle/input/competitions/deep-past-initiative-machine-translation/Sentences_Oare_FirstWord_LinNum.csv
/kaggle/input/competitions/deep-past-initiative-machine-translation/OA_Lexicon_eBL.csv
/kaggle/input/competitions/deep-past-initiative-machine-translation/eBL_Dictionary.csv
/kaggle/input/competitions/deep-past-initiative-machine-translation/train.csv
/kaggle/input/competitions/deep-past-initiative-machine-translation/test.csv
/kaggle/input/competitions/deep-past-initiative-machine-translation/published_texts.csv
/kaggle/input/competitions/deep-past-initiative-machine-translation/resources.csv
/kaggle/input/datasets/llkh0a/byt5-akkadian-model/config.json
/kaggle/input/datasets/llkh0a/byt5-akkadian-model/trainer_state.json
/kaggl

In [2]:
import os
import gc
import re
import json
import math
import random
import logging
import warnings
from pathlib import Path
from dataclasses import dataclass
from contextlib import nullcontext
from typing import List, Tuple, Dict, Optional
from collections import Counter
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, Sampler
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "true")

# ── CONFIG ──────────────────────────────────────────────────────────────────

def _cuda_bf16_supported():
    if not torch.cuda.is_available(): return False
    try: return bool(getattr(torch.cuda, "is_bf16_supported", lambda: False)())
    except: return False

def _bf16_ctx(device, enabled):
    if enabled and device.type == "cuda" and _cuda_bf16_supported():
        return torch.autocast(device_type="cuda", dtype=torch.bfloat16)
    return nullcontext()

@dataclass
class Config:
    test_data_path:  str = "/kaggle/input/competitions/deep-past-initiative-machine-translation/test.csv"
    output_dir:      str = "/kaggle/working/"
    model_a_path:    str = "/kaggle/input/datasets/assiaben/final-byt5/byt5-akkadian-optimized-34x"
    model_b_path:    str = "/kaggle/input/datasets/llkh0a/byt5-akkadian-model"
    model_c_path:    str = "/kaggle/input/datasets/spencevanasperdt/mattiaangelibyt5-akkadian-mbrpytorchdefault6"

    max_input_length:  int   = 512
    max_new_tokens:    int   = 384
    batch_size:        int   = 2
    num_workers:       int   = 2
    num_buckets:       int   = 6
    num_beam_cands:    int   = 6
    num_beams:         int   = 10
    length_penalty:    float = 1.25
    early_stopping:    bool  = True
    repetition_penalty:float = 1.15
    num_sample_cands:  int   = 3
    mbr_top_p:         float = 0.90
    mbr_temperature:   float = 0.70
    mbr_pool_cap:      int   = 48
    use_mixed_precision: bool = True
    checkpoint_freq:   int   = 200

    def __post_init__(self):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        Path(self.output_dir).mkdir(exist_ok=True, parents=True)
        if not torch.cuda.is_available():
            self.use_mixed_precision = False
        self.use_bf16_amp = bool(
            self.use_mixed_precision and
            self.device.type == "cuda" and
            _cuda_bf16_supported()
        )

# ── METRICS ─────────────────────────────────────────────────────────────────

def char_f_score(pred, ref, n=6, beta=2.0):
    pred, ref = pred.strip().lower(), ref.strip().lower()
    if not pred or not ref: return 0.0
    def get_ngrams(text, n):
        if len(text) < n: return Counter([text])
        return Counter([text[i:i+n] for i in range(len(text)-n+1)])
    pred_ng, ref_ng = get_ngrams(pred, n), get_ngrams(ref, n)
    common = sum((pred_ng & ref_ng).values())
    p = common / max(1, sum(pred_ng.values()))
    r = common / max(1, sum(ref_ng.values()))
    if p + r == 0: return 0.0
    return (1 + beta**2) * p * r / (beta**2 * p + r)

def nltk_bleu(pred, ref):
    p, r = pred.strip().lower().split(), ref.strip().lower().split()
    if not p or not r: return 0.0
    try: return sentence_bleu([r], p, smoothing_function=SmoothingFunction().method1)
    except: return 0.0

def geo_score(pred, ref):
    return math.sqrt((char_f_score(pred, ref) + 1e-6) * (nltk_bleu(pred, ref) + 1e-6))

# ── PREPROCESSING ────────────────────────────────────────────────────────────

_V2 = re.compile(r"([aAeEiIuU])(?:2|₂)")
_V3 = re.compile(r"([aAeEiIuU])(?:3|₃)")
_ACUTE = str.maketrans({"a":"á","e":"é","i":"í","u":"ú","A":"Á","E":"É","I":"Í","U":"Ú"})
_GRAVE = str.maketrans({"a":"à","e":"è","i":"ì","u":"ù","A":"À","E":"È","I":"Ì","U":"Ù"})

def ascii_to_diacritics(s):
    s = s.replace("sz","š").replace("SZ","Š")
    s = s.replace("s,","ṣ").replace("S,","Ṣ")
    s = s.replace("t,","ṭ").replace("T,","Ṭ")
    s = _V2.sub(lambda m: m.group(1).translate(_ACUTE), s)
    s = _V3.sub(lambda m: m.group(1).translate(_GRAVE), s)
    return s

OA_HINTS = {
    "KÙ.BABBAR":"silver","KU.BABBAR":"silver","AN.NA":"tin","MA.NA":"mina",
    "GÍN":"shekel","GIN":"shekel","TÚG":"textile","TUG":"textile",
    "GADA":"linen","ŠE":"barley","SE":"barley","URUDU":"copper",
    "ANŠE":"donkey","É":"house","DUMU":"son","DUMU.MUNUS":"daughter",
}

_GAP_RE = re.compile(
    r"<\s*big[\s_\-]*gap\s*>|<\s*gap\s*>|\bbig[\s_\-]*gap\b"
    r"|\bx(?:\s+x)+\b|\.{3,}|…+|\[\.+\]|\[\s*x\s*\]|\(\s*x\s*\)"
    r"|(?<!\w)x{2,}(?!\w)|(?<!\w)x(?!\w)"
    r"|\(\s*large\s+break\s*\)|\(\s*break\s*\)|\(\s*\d+\s+broken\s+lines?\s*\)", re.I
)
_CHAR_TRANS = str.maketrans({
    "ḫ":"h","Ḫ":"H","ʾ":"","₀":"0","₁":"1","₂":"2","₃":"3","₄":"4",
    "₅":"5","₆":"6","₇":"7","₈":"8","₉":"9","—":"-","–":"-",
})
_WS_RE = re.compile(r"\s+")
_FLOAT_RE = re.compile(r"(?<![\w/])(\d+\.\d{4,})(?![\w/])")
_ALLOWED_FRACS = [
    (1/6,"0.16666"),(1/4,"0.25"),(1/3,"0.33333"),(1/2,"0.5"),
    (2/3,"0.66666"),(3/4,"0.75"),(5/6,"0.83333"),
]
def canon_decimal(x):
    ip = int(math.floor(x+1e-12)); frac = x-ip
    best = min(_ALLOWED_FRACS, key=lambda t: abs(frac-t[0]))
    if abs(frac-best[0]) <= 2e-3:
        dec = best[1]
        if ip == 0: return dec
        return f"{ip}{dec[1:]}" if dec.startswith("0.") else f"{ip}+{dec}"
    return f"{x:.5f}".rstrip("0").rstrip(".")

def preprocess(texts: List[str]) -> List[str]:
    results = []
    for text in texts:
        t = ascii_to_diacritics(str(text))
        t = _GAP_RE.sub("<gap>", t)
        t = t.translate(_CHAR_TRANS).replace("ₓ","")
        t = _FLOAT_RE.sub(lambda m: canon_decimal(float(m.group(1))), t)
        t = _WS_RE.sub(" ", t).strip()
        # OA hints
        found = [v for k,v in OA_HINTS.items() if k.upper() in t.upper()]
        if found:
            t += f" [Commodities: {', '.join(list(dict.fromkeys(found))[:4])}]"
        results.append("translate Akkadian to English: " + t)
    return results

# ── POSTPROCESSING ───────────────────────────────────────────────────────────

_EXACT_FRAC_RE = re.compile(r"0\.8333|0\.6666|0\.3333|0\.1666|0\.625|0\.75|0\.25|0\.5")
_EXACT_FRAC_MAP = {"0.8333":"⅚","0.6666":"⅔","0.3333":"⅓","0.1666":"⅙","0.625":"⅝","0.75":"¾","0.25":"¼","0.5":"½"}
_HINT_RE = re.compile(r'\s*\[Commodities:[^\]]*\]', re.I)
_PN_RE = re.compile(r"\bPN\b")
_MULTI_GAP_RE = re.compile(r'(?:<gap>\s*){2,}')
_REPEAT_WORD_RE = re.compile(r"\b(\w+)(?:\s+\1\b)+")
_REPEAT_PUNCT_RE = re.compile(r"([.,])\1+")
_PUNCT_SPACE_RE = re.compile(r"\s+([.,:])")
_FORBIDDEN = str.maketrans("","","()——<>⌈⌋⌊[]+ʾ;")

def postprocess(translations: List[str]) -> List[str]:
    results = []
    for t in translations:
        t = _HINT_RE.sub("", t)
        t = _GAP_RE.sub("<gap>", t)
        t = _PN_RE.sub("<gap>", t)
        t = _EXACT_FRAC_RE.sub(lambda m: _EXACT_FRAC_MAP[m.group(0)], t)
        t = _MULTI_GAP_RE.sub("<gap>", t)
        t = t.replace("<gap>","\x00GAP\x00").translate(_FORBIDDEN).replace("\x00GAP\x00"," <gap> ")
        t = _REPEAT_WORD_RE.sub(r"\1", t)
        t = _PUNCT_SPACE_RE.sub(r"\1", t)
        t = _REPEAT_PUNCT_RE.sub(r"\1", t)
        t = _WS_RE.sub(" ", t).strip()
        results.append(t)
    return results

# ── DATASET ──────────────────────────────────────────────────────────────────

class AkkadianDataset(Dataset):
    def __init__(self, df):
        self.ids = df["id"].tolist()
        self.texts = preprocess(df["transliteration"].tolist())
    def __len__(self): return len(self.ids)
    def __getitem__(self, i): return self.ids[i], self.texts[i]

class BucketSampler(Sampler):
    def __init__(self, dataset, batch_size, num_buckets):
        lengths = [len(t.split()) for _, t in dataset]
        idx = sorted(range(len(lengths)), key=lambda i: lengths[i])
        bsize = max(1, len(idx) // num_buckets)
        self.buckets = [idx[i*bsize:(i+1)*bsize if i<num_buckets-1 else None] for i in range(num_buckets)]
        self.batch_size = batch_size
    def __iter__(self):
        for b in self.buckets:
            for i in range(0, len(b), self.batch_size): yield b[i:i+self.batch_size]
    def __len__(self):
        return sum(math.ceil(len(b)/self.batch_size) for b in self.buckets)

# ── MODEL ────────────────────────────────────────────────────────────────────

class ModelWrapper:
    def __init__(self, path, cfg, label):
        print(f"[{label}] Yükleniyor: {path}")
        self.cfg = cfg
        self.label = label
        self.tok = AutoTokenizer.from_pretrained(path)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(path).to(cfg.device).eval()
        n = sum(p.numel() for p in self.model.parameters())
        print(f"[{label}] {n:,} parametre | GPU: {torch.cuda.memory_allocated()/1e9:.2f} GB")

    def collate(self, batch):
        ids = [b[0] for b in batch]
        texts = [b[1] for b in batch]
        enc = self.tok(texts, max_length=self.cfg.max_input_length, padding=True, truncation=True, return_tensors="pt")
        return ids, enc

    def generate(self, input_ids, attn):
        cfg = self.cfg
        ctx = _bf16_ctx(cfg.device, cfg.use_bf16_amp)
        with ctx:
            beam_out = self.model.generate(
                input_ids=input_ids, attention_mask=attn,
                do_sample=False, num_beams=cfg.num_beams,
                num_return_sequences=cfg.num_beam_cands,
                max_new_tokens=cfg.max_new_tokens,
                length_penalty=cfg.length_penalty,
                repetition_penalty=cfg.repetition_penalty,
                early_stopping=cfg.early_stopping, use_cache=True,
            )
            beam_texts = self.tok.batch_decode(beam_out, skip_special_tokens=True)

            samp_out = self.model.generate(
                input_ids=input_ids, attention_mask=attn,
                do_sample=True, num_beams=1,
                top_p=cfg.mbr_top_p, temperature=cfg.mbr_temperature,
                num_return_sequences=cfg.num_sample_cands,
                max_new_tokens=cfg.max_new_tokens,
                repetition_penalty=cfg.repetition_penalty, use_cache=True,
            )
            samp_texts = self.tok.batch_decode(samp_out, skip_special_tokens=True)

        B = input_ids.shape[0]
        pools = []
        for i in range(B):
            p = list(beam_texts[i*cfg.num_beam_cands:(i+1)*cfg.num_beam_cands])
            p += list(samp_texts[i*cfg.num_sample_cands:(i+1)*cfg.num_sample_cands])
            pools.append(p)
        return pools

    def unload(self):
        del self.model, self.tok
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            free = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9
            print(f"[{self.label}] Kaldırıldı. GPU boş: {free:.2f} GB")

# ── MBR ──────────────────────────────────────────────────────────────────────

def mbr_pick(candidates: List[str], pool_cap=48) -> str:
    seen, cands = set(), []
    for c in candidates:
        c = str(c).strip()
        if c and c not in seen:
            cands.append(c); seen.add(c)
    cands = cands[:pool_cap]
    n = len(cands)
    if n == 0: return ""
    if n == 1: return cands[0]
    best_i, best_s = 0, -1e9
    for i in range(n):
        s = sum(geo_score(cands[i], cands[j]) for j in range(n) if j != i) / max(1, n-1)
        if s > best_s: best_s, best_i = s, i
    return cands[best_i]

# ── MAIN ──────────────────────────────────────────────────────────────────────

cfg = Config()
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"BF16: {cfg.use_bf16_amp}")

test_df = pd.read_csv(cfg.test_data_path)
print(f"Test örnekleri: {len(test_df)}")

dataset = AkkadianDataset(test_df)
all_pools = {str(sid): [] for sid in dataset.ids}

for label, path in [
    ("Model-A", cfg.model_a_path),
    ("Model-B", cfg.model_b_path),
    ("Model-C", cfg.model_c_path),
]:
    wrapper = ModelWrapper(path, cfg, label)
    sampler = BucketSampler(dataset, cfg.batch_size, cfg.num_buckets)
    dl = DataLoader(dataset, batch_sampler=sampler, num_workers=cfg.num_workers,
                    collate_fn=wrapper.collate, pin_memory=True)

    with torch.inference_mode():
        for batch_ids, enc in tqdm(dl, desc=label):
            input_ids = enc.input_ids.to(cfg.device)
            attn = enc.attention_mask.to(cfg.device)
            try:
                pools = wrapper.generate(input_ids, attn)
                for sid, pool in zip(batch_ids, pools):
                    all_pools[str(sid)].extend(pool)
            except RuntimeError as e:
                if "out of memory" in str(e).lower():
                    print(f"OOM! Batch atlandı.")
                    torch.cuda.empty_cache()
                else:
                    raise

    wrapper.unload()

print("MBR seçimi yapılıyor...")
results = []
for sid in [str(s) for s in dataset.ids]:
    candidates = all_pools[sid]
    pp = postprocess(candidates)
    chosen = mbr_pick(pp, cfg.mbr_pool_cap)
    if not chosen.strip():
        chosen = "The tablet is too damaged to translate."
    results.append((sid, chosen))

result_df = pd.DataFrame(results, columns=["id", "translation"])

out_path = Path(cfg.output_dir) / "submission.csv"
result_df.to_csv(out_path, index=False)

print(f"\n✅ Tamamlandı! {len(result_df)} satır → {out_path}")
print(result_df)


GPU: Tesla P100-PCIE-16GB
BF16: True
Test örnekleri: 4
[Model-A] Yükleniyor: /kaggle/input/datasets/assiaben/final-byt5/byt5-akkadian-optimized-34x


Loading weights:   0%|          | 0/252 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


[Model-A] 581,653,248 parametre | GPU: 2.38 GB


Model-A:   0%|          | 0/4 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/252 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


[Model-C] 581,653,248 parametre | GPU: 2.39 GB


Model-C:   0%|          | 0/4 [00:00<?, ?it/s]

[Model-C] Kaldırıldı. GPU boş: 17.05 GB
MBR seçimi yapılıyor...

✅ Tamamlandı! 4 satır → /kaggle/working/submission.csv
  id                                        translation
0  0  Thus kārum Kanesh, say to the <gap> dātum, our...
1  1  In the tablet of the City, you wrote to me in ...
2  2  As soon as you hear our letter, there he has g...
3  3  I sent it to every single colony and the tradi...
